# 05 Alarm Triage Layer

This notebook explains the operator-oriented alarm triage layer. The models produce timestamp-level predictions, but an operator should not receive one card for every positive timestamp. The triage script groups consecutive positive predictions into alarm episodes, then combines model score, threshold margin, model agreement, temporal behavior, and XAI evidence quality into a more useful review summary.

The emphasis is operational: which alarms look strong or weak, whether evidence is recent and concentrated, whether multiple models agree, and how the alarm should be prioritized for human checking. Labels are used here for offline evaluation summaries, not as real-time inputs to the reliability or risk scores.


## Load Predictions, Episodes, and All-Alarm XAI

This cell loads the saved final predictions, grouped alarm episodes, and full XAI evidence tables for all alarm episode representative rows.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "nita_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "nita_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))
from nita_xai import config

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

tables_dir = config.TABLES_DIR
predictions = pd.read_csv(config.FINAL_PREDICTIONS_DIR / "all_model_predictions.csv")
incidents = pd.read_csv(config.INCIDENT_EPISODE_SUMMARY_PATH)
episodes = pd.read_csv(config.ALARM_EPISODE_TRIAGE_PATH)
false_alarms = pd.read_csv(config.FALSE_ALARM_TRIAGE_PATH)
targets = pd.read_csv(tables_dir / "xai_alarm_episode_targets.csv")
evidence = pd.read_csv(tables_dir / "xai_alarm_evidence_metrics.csv")
method_agreement = pd.read_csv(tables_dir / "xai_method_agreement_all_alarm_episodes.csv")

print(f"Timestamp predictions: {len(predictions):,}")
print(f"Incident streams: {len(incidents):,}")
print(f"Alarm episodes: {len(episodes):,}")
print(f"False alarm episodes: {len(false_alarms):,}")
print(f"XAI alarm representative targets: {len(targets):,}")
print(f"XAI evidence rows: {len(evidence):,}")


**Interpretation:** timestamp predictions are useful for metrics, but the operator-facing layer works at episode level. The XAI evidence rows explain the representative timestamp of each episode, usually the peak-score moment.


## Timestamp-Level Prediction Outcomes

This is the normal machine-learning view. It counts every 5-minute row separately.


In [ ]:
prediction_summary = predictions.groupby(["model", "case_type"]).size().unstack(fill_value=0)
prediction_summary


**Interpretation:** this table is useful for offline evaluation, but it is too granular for traffic control work. A single sustained alarm could create many positive timestamps, so we group them into episodes.


## Episode-Level Alarm Outcomes

This is the operator-oriented view. Consecutive positive timestamps from the same model and stream become one alarm episode.


In [ ]:
episode_summary = episodes.groupby(["model", "evaluation_episode_type"]).agg(
    episodes=("alarm_episode_id", "count"),
    mean_duration_min=("alarm_duration_minutes", "mean"),
    mean_peak_probability=("peak_probability", "mean"),
    mean_threshold_margin=("mean_threshold_margin", "mean"),
).reset_index()
episode_summary


**Interpretation:** episode counts are closer to operator workload than timestamp counts. Duration and threshold margin help separate brief borderline alarms from stronger sustained alarms.


## Reliability and Risk Indicators

Reliability and false-positive risk are based on model behavior and context indicators such as score margin, persistence, model agreement, and explanation quality. These scores do not use the ground-truth label as a real-time input.


In [ ]:
triage_distribution = episodes.groupby([
    "model", "reliability_level", "false_positive_risk_level", "operator_priority"
]).size().reset_index(name="episode_count")
triage_distribution.sort_values(["model", "episode_count"], ascending=[True, False]).head(25)


**Interpretation:** this table describes how much review burden the alarm system creates and how those alarms might be prioritized. It should be read as support for operator judgment, not as an automatic truth decision.


## XAI Evidence Quality Across Alarm Episodes

This summary joins the triage idea with all-alarm XAI. It shows whether representative alarm explanations are recent, concentrated, and balanced between speed and occupancy.


In [ ]:
evidence_quality = evidence.groupby(["model", "method"]).agg(
    cases=("case_id", "nunique"),
    recent_evidence_mean=("recent_0_15_min_share", "mean"),
    top3_concentration_mean=("top3_attribution_concentration", "mean"),
    speed_share_mean=("speed_attribution_share", "mean"),
    occupancy_share_mean=("occupancy_attribution_share", "mean"),
    diffuseness_mean=("explanation_diffuseness", "mean"),
).reset_index()
evidence_quality


**Interpretation:** recent and concentrated explanations are easier to summarize for an operator. Diffuse explanations can still be valid, but they are weaker as a concise alarm rationale because the evidence is spread across more lagged inputs.


## Method Agreement as a Triage Signal

For models with two explanation methods, SHAP/LIME-style agreement adds another evidence-quality signal. Agreement means the methods highlight similar lagged traffic features for the same alarm representative row.


In [ ]:
method_agreement_summary = method_agreement.groupby([
    "model", "explanation_agreement_level"
]).size().unstack(fill_value=0)
method_agreement_summary


**Interpretation:** agreement does not prove an alarm is physically real. It means the explanation is more stable, which can make the alarm easier to present and review.


## False Alarm Types

These are after-the-fact evaluation categories for alarms that did not overlap labeled incident timestamps. They are useful for analysis, but the real-time triage layer should still describe the evidence rather than tell the operator it is false.


In [ ]:
false_alarm_types = false_alarms.groupby(["model", "alarm_type"]).agg(
    episodes=("alarm_episode_id", "count"),
    mean_duration_min=("alarm_duration_minutes", "mean"),
    mean_peak_probability=("peak_probability", "mean"),
    mean_threshold_margin=("mean_threshold_margin", "mean"),
).reset_index().sort_values(["model", "episodes"], ascending=[True, False])
false_alarm_types.head(25)


**Interpretation:** false alarms are not all the same. A short one-model spike should be handled differently from a persistent multi-model consensus alarm, even if both are counted as false positives during offline evaluation.


## Example Incident Alarms

These examples overlap labeled incident timestamps during offline evaluation. They show how the episode table combines alarm strength, model agreement, XAI evidence, and temporal pattern.


In [ ]:
example_cols = [
    "model", "stream_id", "alarm_start", "alarm_end", "evaluation_episode_type",
    "peak_probability", "mean_threshold_margin", "model_agreement_count",
    "dominant_xai_features", "explanation_agreement_level", "temporal_pattern",
    "reliability_level", "false_positive_risk_level", "operator_priority", "operator_note",
]
episodes[episodes["evaluation_episode_type"] == "true_incident_alarm"].sort_values(
    ["model_agreement_count", "peak_probability"], ascending=[False, False]
)[example_cols].head(10)


**Interpretation:** these rows show the intended operator-support format: not only whether the model alarmed, but how strong, persistent, and explainable the alarm was.


## Example Review-Worthy Alarms Outside Labeled Incidents

These examples did not overlap the target incident label in offline evaluation. In real time, the operator would not know that label, so the wording should focus on evidence quality and suggested review priority.


In [ ]:
false_alarms.sort_values(
    ["model_agreement_count", "peak_probability"], ascending=[False, False]
)[example_cols].head(10)


**Interpretation:** this table helps separate high-confidence review-worthy alarms from lower-priority alarms. It should not be used to claim XAI can prove whether the physical cause was an incident.


## Operator-Facing Alarm Case Studies

The following two examples are retained as readable illustrations. The first is a strong multi-model alarm where the evidence is concentrated in recent traffic context. The second is a weaker borderline alarm where the score is only slightly above threshold and fewer models agree.


In [ ]:
case_dir = config.RESULTS_DIR / "case_studies"
strong_alarm_card = (case_dir / "false_alarm_stream_50_operator_card.txt").read_text(
    encoding="utf-8"
)
print(strong_alarm_card)


The next figure shows the local speed and occupancy context around the strong alarm. The dashed vertical line marks the alarm timestamp.

![Strong alarm traffic context](../results/figures/case_studies/false_alarm_stream_50_context.png)


In [ ]:
strong_alarm_xai = pd.read_csv(case_dir / "false_alarm_stream_50_xai_evidence.csv")
strong_alarm_xai[[
    "model", "rank", "feature", "traffic_signal", "minutes_before_alarm",
    "direction", "abs_attribution",
]].head(12)


The next figure translates the XGBoost local explanation into a readable contribution chart. Blue bars are speed evidence and red bars are occupancy evidence.

![Strong alarm XGBoost XAI evidence](../results/figures/case_studies/false_alarm_stream_50_xgboost_xai.png)


**Operator interpretation:** from an operator viewpoint, this alarm is review-worthy because the models detected abnormal-looking local traffic context. The strongest evidence is concentrated around the last 5-15 minutes before the alarm, especially recent speed behavior, with supporting occupancy context. This does not prove an incident, but it gives the operator a reason to check camera, map, or sensor conditions.


### Weak / Borderline Alarm Case

This case is intentionally different. The score is only slightly above the threshold, fewer models agree, and the explanation is less useful as a strong alarm rationale.


In [ ]:
borderline = pd.read_csv(config.TABLES_DIR / "borderline_alarm_xai_case_study.csv")
borderline[[
    "model", "fold", "stream_id", "timestamp", "anomaly_score", "threshold",
    "score_margin", "model_agreement_count", "operator_priority",
]].drop_duplicates()


In [ ]:
borderline[[
    "rank", "feature", "traffic_signal", "minutes_before_alarm", "direction",
    "abs_attribution", "top3_attribution_share", "recent_0_15_min_share_top10",
    "mid_15_30_min_share_top10", "older_30_60_min_share_top10",
]].head(10)


![Borderline alarm XAI evidence](../results/figures/xai/borderline_alarm_case_study.png)


**Operator interpretation:** although this is technically an alarm, the evidence is weaker than the previous case. The score is only slightly above the threshold, fewer models agree, and the explanation should be treated as lower-priority support unless external evidence confirms worsening traffic conditions.


### Developing-Pattern Alarm Case

The previous cases show strong and borderline alarm profiles. This case shows another useful operator-facing idea: an alarm can be influenced by traffic evidence that started earlier in the 60-minute lookback window, not only by the last 5 minutes. That helps the operator understand whether the alarm looks sudden or whether the model is reacting to a developing pattern.


In [ ]:
local_xai = pd.read_csv(tables_dir / "xai_local_explanations_all_alarm_episodes.csv")

developing_case_id = "fold_2_xgboost_202548"
developing_method = "lime_style_local_surrogate"

developing_case = (
    local_xai[
        (local_xai["case_id"] == developing_case_id)
        & (local_xai["method"] == developing_method)
    ]
    .sort_values("rank")
    .copy()
)

developing_target = targets[
    (targets["model"] == "xgboost")
    & (targets["representative_row_id"] == int(developing_case["row_id"].iloc[0]))
]

developing_episode = episodes[
    (episodes["model"] == "xgboost")
    & (episodes["representative_row_id"] == int(developing_case["row_id"].iloc[0]))
]

developing_evidence = evidence[
    (evidence["case_id"] == developing_case_id)
    & (evidence["method"] == developing_method)
]

summary_cols = [
    "model", "fold", "stream_id", "source_file", "alarm_start", "alarm_end",
    "representative_timestamp", "peak_probability", "max_threshold_margin",
    "model_agreement_count", "evaluation_episode_type", "operator_priority",
]
developing_episode[summary_cols]


**Interpretation:** this is an offline case study. The `evaluation_episode_type` column is useful for analysis after the fact, but an operator would not know that label at alarm time. The operator-facing part is the score margin, model agreement, alarm duration, and XAI evidence pattern.


In [ ]:
developing_case[[
    "rank", "feature", "traffic_signal", "minutes_before_alarm",
    "attribution", "direction", "recency_bucket",
]].head(10)


In [ ]:
developing_evidence[[
    "speed_attribution_share", "occupancy_attribution_share",
    "recent_0_15_min_share", "mid_15_30_min_share",
    "older_30_60_min_share", "top3_attribution_concentration",
]]


In [ ]:
def readable_feature_name(feature):
    signal = "Speed" if feature.startswith("speed") else "Occupancy"
    lag = int(feature.rsplit("_", 1)[-1])
    return f"{signal}, {lag * 5} min before"

plot_case = developing_case.nsmallest(10, "rank").copy()
plot_case["readable_feature"] = plot_case["feature"].map(readable_feature_name)
plot_case = plot_case.sort_values("attribution")

colors = plot_case["traffic_signal"].map({"speed": "#2C7FB8", "occupancy": "#D7301F"})
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(plot_case["readable_feature"], plot_case["attribution"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("LIME-style local attribution")
ax.set_title("Developing-pattern alarm evidence: XGBoost, stream 172")
plt.tight_layout()
fig


**Operator interpretation:** this alarm is useful because the explanation is not only based on the final 5 minutes before the representative timestamp. In this XGBoost case, the LIME-style explanation places most of the evidence in the older part of the 60-minute window, especially speed values from around 45-60 minutes earlier. This suggests the model was reacting to a developing traffic pattern rather than only a sudden last-minute change. This does not prove the physical cause, but it gives the operator context for reviewing camera, map, or sensor conditions.


## Operator Card Examples

These cards show how a model output can be translated into concise operator support. They are examples of the final layer, not replacements for professional traffic judgment.


In [ ]:
def render_operator_card(row):
    lines = [
        f"Alarm episode {row.alarm_episode_id} ({row.model})",
        f"Stream: {row.stream_id}",
        f"Time: {row.alarm_start} to {row.alarm_end}",
        f"Peak score: {row.peak_probability:.3f}",
        f"Model agreement: {row.model_agreement_count}",
        f"Reliability: {row.reliability_level}",
        f"False-positive risk: {row.false_positive_risk_level}",
        f"Priority: {row.operator_priority}",
        f"Evidence: {row.dominant_xai_features}",
        f"Note: {row.operator_note}",
    ]
    return "\n".join(lines)

for _, row in episodes.sort_values(
    ["operator_priority", "model_agreement_count", "peak_probability"],
    ascending=[True, False, False],
).head(3).iterrows():
    print(render_operator_card(row))
    print("\n" + "-" * 80 + "\n")


**Final interpretation:** XAI is used as an alarm evidence-quality layer. Score margin and model agreement show how strongly the system alarmed, while SHAP/LIME/DeepSHAP show whether the alarm evidence was recent, concentrated, and consistent. Because XAI was computed for all alarm episode representative rows across the five folds, the resulting patterns describe the alarm set rather than isolated examples.
